# Step 4 – TTS: অনুবাদ থেকে বাংলা আর ইংরেজি কণ্ঠ (MMS-TTS)
**Runtime → Change runtime type → T4 GPU**, তারপর **Run all**। GPU না থাকলেও চলে, শুধু ধীর।
কোনো ট্রেনিং নেই: Meta-র আগে থেকে তৈরি `facebook/mms-tts-ben` আর `facebook/mms-tts-eng` ব্যবহার হয়।
মূল্যায়ন: MT-র ১০০টা অনুবাদ কণ্ঠে বানিয়ে আবার Whisper দিয়ে লেখা হয় (বোধগম্যতা = CER/WER), সাথে গতি (RTF)।
ফল সেভ হয় Drive এর `sylheti_tts` ফোল্ডারে।

## 1. Drive আর library

In [ ]:
from google.colab import drive
import os, sys, subprocess
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
N = 100              # how many test sentences to synthesise and evaluate
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.48.3", "accelerate==1.3.0",
                "jiwer==3.0.5", "soundfile", "uroman"], check=True)
import torch
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(0) if dev == "cuda" else "CPU (slow but works)")

## 2. MT-র অনুবাদ খুঁজে আনা

In [ ]:
import pandas as pd
D = "/content/drive/MyDrive/Sylheti_Thesis"
cands = [D + "/sylheti_mt_copy/nllb-sylheti", D + "/sylheti_mt/nllb-sylheti"]
MT_RUN = next((p for p in cands if os.path.exists(p + "/predictions_test.csv")), None)
assert MT_RUN, "predictions_test.csv পাওয়া যায়নি: sylheti_mt (বা sylheti_mt_copy) My Drive-এ আছে কিনা দেখুন"
print("MT results from:", MT_RUN)
pred = pd.read_csv(MT_RUN + "/predictions_test.csv", encoding="utf-8-sig")
pred = pred.dropna(subset=["pred_bn", "pred_en"]).sample(min(N, len(pred)), random_state=13).reset_index(drop=True)
OUT = D + "/sylheti_tts"; os.makedirs(OUT + "/wav_bn", exist_ok=True); os.makedirs(OUT + "/wav_en", exist_ok=True)
print(len(pred), "sentences | example:", pred.pred_bn[0], "|", pred.pred_en[0])

## 3. কণ্ঠ বানানো (MMS-TTS)

In [ ]:
import time, re, numpy as np, soundfile as sf
from transformers import VitsModel, AutoTokenizer
try:
    import uroman as ur; _uroman = ur.Uroman()
except Exception:
    _uroman = None
tts = {}
for k, name in [("bn", "facebook/mms-tts-ben"), ("en", "facebook/mms-tts-eng")]:
    tts[k] = (AutoTokenizer.from_pretrained(name), VitsModel.from_pretrained(name).to(dev).eval())

def speak(text, lang, seed=0):
    tok, m = tts[lang]
    text = re.sub(r"\s+", " ", str(text)).strip()
    if getattr(tok, "is_uroman", False) and _uroman is not None:
        text = _uroman.romanize_string(text)
    if lang == "en":
        text = text.lower()
    ids = tok(text, return_tensors="pt").to(dev)
    torch.manual_seed(seed)                      # same text -> same voice every time
    with torch.no_grad():
        wav = m(**ids).waveform[0].cpu().numpy()
    return wav, m.config.sampling_rate

rows = []
for i, r in pred.iterrows():
    for k in ["bn", "en"]:
        t0 = time.time(); wav, sr = speak(r["pred_" + k], k); dt = time.time() - t0
        path = f"{OUT}/wav_{k}/{i:03d}.wav"; sf.write(path, wav, sr)
        rows.append({"id": i, "lang": k, "text": r["pred_" + k], "wav": path, "sec": len(wav) / sr, "gen_sec": dt})
tt = pd.DataFrame(rows); tt.to_csv(OUT + "/tts_outputs.csv", index=False, encoding="utf-8-sig")
for k in ["bn", "en"]:
    s = tt[tt.lang == k]
    print(f"{k}: {len(s)} clips, {s.sec.sum()/60:.1f} min audio, RTF {s.gen_sec.sum()/s.sec.sum():.3f} (lower = faster)")

## 4. মূল্যায়ন: তৈরি কণ্ঠ আবার Whisper দিয়ে লেখা (বোধগম্যতা)

In [ ]:
import jiwer, unicodedata, librosa
from transformers import pipeline
asr = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3-turbo",
               torch_dtype=torch.float16 if dev == "cuda" else torch.float32, device=0 if dev == "cuda" else -1)
def norm(s, lang):
    s = unicodedata.normalize("NFC", str(s)).lower()
    s = re.sub(r"[\"\'“”‘’(),.।?!:;\-/*…\[\]|]", " ", s)
    return re.sub(r"\s+", " ", s).strip()
res = []
for k, lang in [("bn", "bengali"), ("en", "english")]:
    s = tt[tt.lang == k].reset_index(drop=True)
    audio = [librosa.load(p, sr=16000)[0] for p in s.wav]
    out = asr(audio, batch_size=8, generate_kwargs={"language": lang, "task": "transcribe", "max_new_tokens": 200})
    hyp = [norm(o["text"], k) or "-" for o in out]; ref = [norm(t, k) for t in s.text]
    tt.loc[tt.lang == k, "asr_back"] = hyp
    res.append({"language": k, "clips": len(s), "CER %": round(100 * jiwer.cer(ref, hyp), 1),
                "WER %": round(100 * jiwer.wer(ref, hyp), 1),
                "audio min": round(s.sec.sum() / 60, 1), "RTF": round(s.gen_sec.sum() / s.sec.sum(), 3)})
res = pd.DataFrame(res); res.to_csv(OUT + "/results.csv", index=False)
tt.to_csv(OUT + "/tts_outputs.csv", index=False, encoding="utf-8-sig")
print(res.to_string(index=False))
print("\nsaved:", OUT + "/results.csv")

## 5. শুনে দেখুন

In [ ]:
from IPython.display import Audio, display
import random
ids = random.sample(sorted(tt.id.unique()), 3)   # new random sentences every run
for i in ids:
    print(f"---- বাক্য #{i} ----")
    for k in ["bn", "en"]:
        r = tt[(tt.id == i) & (tt.lang == k)].iloc[0]
        print(k, "| লেখা:", r.text)
        print("   | Whisper শুনেছে:", r.asr_back)
        display(Audio(r.wav))
    print()

## 6. নিজে লিখে শুনুন

In [ ]:
text_bn = "আমি এখন বাড়ি যাব"     # change these
text_en = "I will go home now"
for t, k in [(text_bn, "bn"), (text_en, "en")]:
    wav, sr = speak(t, k); print(k, t); display(Audio(wav, rate=sr))